# Eksperimen 1 — kandidat CAD-ViT awal (V1)

**Pertanyaan penelitian:** seberapa jauh kandidat model dan preprocessing yang diturunkan dari uraian CAD-ViT dapat melakukan grading DR lima kelas pada APTOS2019 dan DDR? Ini catatan **historis** untuk run `aptos_main_80ep_seed42` dan `ddr_main_80ep_seed42`, bukan kode pelatihan aktif. [Ringkasan utama](../../CAD_ViT.ipynb) · [notebook asli 105 sel](../archive/CAD_ViT_original_105_cells.ipynb). Tanggal eksekusi V1 tidak dapat dipastikan dari artefak lokal.

## Dataset, protokol, dan cakupan bukti

APTOS2019 berasal dari pasangan `train.csv`/`train_images` (3.662 pasangan berlabel); DDR grading menggunakan 12.522 anotasi grade 0–4 dari sumber OIA-DDR. Keduanya memakai split reproduksi 8:1:1 berkelompok SHA-256, seed 42; manifest beku dan checkpoint tidak tersedia lokal. Ringkasan V1 mencatat manifest SHA-256 APTOS `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` dan DDR `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1`; hash ini adalah metadata historis, bukan verifikasi isi manifest lokal. Test hasil historis mencakup **366 APTOS** dan **1.252 DDR**. Jumlah train unik yang tercatat pada config generasi terbaru adalah 2.930 APTOS dan 10.018 DDR, namun isi manifest V1 tidak dapat diaudit lokal sehingga angka ini tidak dijadikan bukti split V1 yang identik.

Preprocessing V1: EXIF/RGB → crop border gelap (ambang 10) → resize `INTER_AREA` 512×512 → Gaussian 5×5, σ=1. Flip/shear/rotation hanya untuk train; `WeightedRandomSampler` inverse-frequency hanya pada train. Target paper: 80 epoch, LR 0,000186, batch fisik 16, reduction 16. AdamW, weight decay 0, cross-entropy, tanpa scheduler/pretraining, mixed precision opsional, dan seleksi checkpoint validation macro-F1 (tie: loss) adalah **asumsi implementasi** pada [arsip sel 65–66](../archive/CAD_ViT_original_105_cells.ipynb); konfigurasi runtime V1 tepatnya tidak dapat dibuktikan ulang dari checkpoint eksternal. Model kandidat 5.282.101 parameter versus 7.262.900 yang dilaporkan paper.

## Walkthrough implementasi historis

| Langkah | Source historis | Input → output dan peran |
| --- | --- | --- |
| Baca data | [Arsip sel 41: `aptos_records`, `ddr_records`](../archive/CAD_ViT_original_105_cells.ipynb) | Root dataset/anotasi → rekaman gambar dan label; kelas DDR 5 ungradable dikecualikan. |
| Manifest | Arsip sel 45: `_aptos_group_split`, `prepare_manifest` | Rekaman + seed → split berkelompok/hash; manifest asli berada di luar repo. |
| Transform dan loader | Arsip sel 41: `preprocess_fundus`; sel 47: `FundusDataset`, `make_loaders` | Gambar/label → tensor 512×512 dan batch; augmentasi serta `WeightedRandomSampler` pada train. Implementasi reusable terpisah: [`preprocessing.py`](../cadvit/preprocessing.py). |
| Arsitektur | Arsip sel 53/55: `CoordinateAttention`, `MultiScaleFeatureBlock`, `CADViTCandidate` | Tensor RGB → lima logit. Definisi model reusable cocok secara AST dengan [`model.py`](../cadvit/model.py). |
| Training | Arsip sel 66: `ExperimentConfig`, `run_experiment`, `_train_epoch`, `score_loader` | Manifest + konfigurasi → history dan `best.pt` menurut validation macro-F1; checkpoint disimpan di luar repo. |
| Evaluasi historis | Arsip sel 85/88 | Checkpoint terpilih + split test → ringkasan dan matriks tersimpan. Ini bukti V1; kode tidak dieksekusi ulang di sini. |

Modul [`experiments.py`](../cadvit/experiments.py) adalah runner modular generasi selanjutnya; nama fungsi yang mirip tidak membuktikan kesetaraan protokol dengan runner V1. Source yang sama juga digunakan untuk eksperimen V2 historis dengan fungsi baru di arsip sel 68.

## Riwayat training yang tersedia

| Dataset | Rekaman epoch yang tersedia | Seleksi checkpoint menurut laporan historis | Batas |
| --- | --- | --- | --- |
| APTOS2019 | Log epoch **1–43** dilaporkan; baris log mentah tidak tersedia di repo | Epoch **8**, validation macro-F1 sekitar **0,580661** | Nama run `80ep` tidak membuktikan 80 epoch selesai. |
| DDR | Segmen resume epoch **59–69** dilaporkan; train/validation loss NaN di segmen itu | Epoch **5** menurut ringkasan final test; metrik validation epoch itu tidak tersedia | Awal/penyebab instabilitas dan riwayat epoch lain belum terverifikasi. |

Karena baris sumber lengkap tidak berada dalam workspace, tabel epoch dan kurva V1 tidak direkonstruksi. Ringkasan aslinya ada di [arsip §5.1 dan §5.8](../archive/CAD_ViT_original_105_cells.ipynb). Hasil final test yang sudah tercatat tetap dapat dibaca tanpa menjalankan evaluasi.

## Final test historis V1

| Dataset/run | Checkpoint dari validation | N test | Accuracy test | Macro-F1 test | Macro specificity test | Sumber |
| --- | ---: | ---: | ---: | ---: | ---: | --- |
| APTOS `aptos_main_80ep_seed42` | 8 | 366 | 0,724044 | 0,554058 | 0,933978 | [Arsip §6.2–6.3](../archive/CAD_ViT_original_105_cells.ipynb); JSON mentah eksternal, SHA yang dicatat `3cba5fb3fdd9554ec35b876d83512d207570f23d9227082c3acfd50865b4b03d` |
| DDR `ddr_main_80ep_seed42` | 5 | 1.252 | 0,431310 | 0,353457 | 0,857445 | Arsip §6.2–6.3; JSON mentah eksternal, SHA yang dicatat `b90a19cfaa68f5f0bebfedff76703825dd851cbf0362afb440fb1b4f7bf7e95e` |

Precision/recall per kelas dan support berikut dihitung dari matriks tersimpan; presisi angka agregat di atas mengikuti tabel historis. JSON asal tidak dapat dihitung hash ulang di workspace ini.

### V1 historis APTOS — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 170 | 7 | 2 | 1 | 0 | 180 |
| 1 | 2 | 25 | 10 | 1 | 0 | 38 |
| 2 | 4 | 16 | 52 | 26 | 2 | 100 |
| 3 | 0 | 1 | 3 | 10 | 5 | 19 |
| 4 | 0 | 1 | 7 | 13 | 8 | 29 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 180 | 0.965909 | 0.944444 | 0.955056 | 0.967742 |
| 1 | 38 | 0.500000 | 0.657895 | 0.568182 | 0.923780 |
| 2 | 100 | 0.702703 | 0.520000 | 0.597701 | 0.917293 |
| 3 | 19 | 0.196078 | 0.526316 | 0.285714 | 0.881844 |
| 4 | 29 | 0.533333 | 0.275862 | 0.363636 | 0.979228 |

![Confusion matrix V1 APTOS](../../reproduction/report_figures/historical_v1_aptos_confusion_matrix.png)

### V1 historis DDR — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 289 | 238 | 70 | 0 | 29 | 626 |
| 1 | 8 | 37 | 12 | 1 | 5 | 63 |
| 2 | 79 | 124 | 152 | 17 | 76 | 448 |
| 3 | 0 | 4 | 10 | 5 | 4 | 23 |
| 4 | 3 | 2 | 18 | 12 | 57 | 92 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 626 | 0.762533 | 0.461661 | 0.575124 | 0.856230 |
| 1 | 63 | 0.091358 | 0.587302 | 0.158120 | 0.690496 |
| 2 | 448 | 0.580153 | 0.339286 | 0.428169 | 0.863184 |
| 3 | 23 | 0.142857 | 0.217391 | 0.172414 | 0.975590 |
| 4 | 92 | 0.333333 | 0.619565 | 0.433460 | 0.901724 |

![Confusion matrix V1 DDR](../../reproduction/report_figures/historical_v1_ddr_confusion_matrix.png)

## Interpretasi, keterbatasan, dan langkah berikut

Pada APTOS, grade 2→3 terjadi 26 kali; pada DDR, grade 0→1 terjadi 238 kali dan grade 2→1 124 kali. Ini deskripsi hitungan dari test, bukan diagnosis klinis. Kelas minoritas memiliki support kecil. Hasil DDR V1 yang rendah dan instabilitas numerik pada log resume menjadi salah satu alasan meneliti preprocessing/optimisasi V2; pengaruh kausal perubahan tertentu tidak dapat disimpulkan dari perbandingan lintas run.

**Provenance:** source dan output notebook V1 dipertahankan byte demi byte dalam [arsip](../archive/CAD_ViT_original_105_cells.ipynb). Manifest V1, file `aptos_test.json`/`ddr_test.json`, checkpoint, log epoch mentah, dan transkrip eksekusi berada di luar repo. Source hash JSON di tabel adalah catatan historis, bukan verifikasi byte lokal. Jangan memakai checkpoint atau kode evaluator V2 sebagai pengganti V1. Lanjut ke [eksperimen 2 — V2 BG10](02_experiment_bg10_cadvit_v2.ipynb).